## Reduced without AMPA, Wong and Wang 2006

In [ ]:
# 这个实验中，我通过设置不同的coherent系数，得到了复杂的综合图，符合预期。
# 随后，我补充了决策模块，绘制出了相应的图。

In [21]:
# Import necessary packages
using Random
using Statistics
using PlotlyJS

# Set random seed for reproducibility
Random.seed!(1234)

# Simulation Parameters
dt = 0.5                        # Time step (ms)
t_max = 3000.0                  # Total simulation time (ms)
time = 0.0:dt:t_max             # Time vector
n_time_steps = length(time)

# Model Parameters
# Time Constants
τ_NMDA = 100.0                  # NMDA synaptic time constant (ms)
τ_AMPA = 2.0                    # AMPA synaptic time constant (ms)

# Synaptic Efficacies
J_N = 0.2609                    # Recurrent NMDA excitation strength
J_N_cross = 0.0497              # Cross NMDA excitation

# NMDA Gain Factor
γ = 0.641                       # NMDA receptor gain factor

# External Inputs
I_0 = 0.3255                    # Baseline external input (nA)
μ0 = 30                         # Hz, mean input
J_Input = 5.2e-4                # Input excitation
coherence_values = [0.0, 0.128, 0.512]  # Different coherence levels

stim_start = 500.0              # Stimulus onset time (ms)
stim_end = 2000.0               # Stimulus offset time (ms)

# Noise Parameters
σ_I = 0.02                      # Reduced standard deviation of input noise (nA)

# Define transfer function H(I)
# F-I Curve Parameters (Excitatory Neurons)
a = 270.0                       # (Hz/nA)
b = 108.0                       # (Hz)
d = 0.154                       # (s)
function H(I)
    I_eff = a * I - b
    return I_eff / (1.0 - exp(-d * I_eff))
end

# Function to run simulation for a given coherence
function run_simulation(coherence)
    # Stimulus input to Population 1&2
    I_stim_1 = J_Input * μ0 * (1 + coherence) 
    I_stim_2 = J_Input * μ0 * (1 - coherence)

    # Initialize variables
    S1 = zeros(n_time_steps)        # Synaptic gating variable for Population 1
    S2 = zeros(n_time_steps)        # Synaptic gating variable for Population 2

    r1 = zeros(n_time_steps)        # Firing rate of Population 1
    r2 = zeros(n_time_steps)        # Firing rate of Population 2

    I1 = zeros(n_time_steps)        # Total input current to Population 1
    I2 = zeros(n_time_steps)        # Total input current to Population 2

    # Precompute external inputs
    I_ext1 = fill(I_0, n_time_steps)
    I_ext2 = fill(I_0, n_time_steps)

    stim_indices = findall(t -> t >= stim_start && t <= stim_end, time)
    I_ext1[stim_indices] .+= I_stim_1
    I_ext2[stim_indices] .+= I_stim_2

    # Simulate the network dynamics
    I_noise1 = σ_I * randn()
    I_noise2 = σ_I * randn()

    S1[1] = S2[1] = 0.1
    for t in 1:n_time_steps-1
        # Compute noise terms
        I_noise1 = I_noise1 + dt / τ_AMPA * (-I_noise1) + sqrt(dt/τ_AMPA) * σ_I * randn()
        I_noise2 = I_noise2 + dt / τ_AMPA * (-I_noise2) + sqrt(dt/τ_AMPA) * σ_I * randn()

        # Compute total input currents
        I1[t] = J_N * S1[t] - J_N_cross * S2[t] + I_ext1[t] + I_noise1
        I2[t] = J_N * S2[t] - J_N_cross * S1[t] + I_ext2[t] + I_noise2

        # Compute firing rates
        r1[t] = H(I1[t])
        r2[t] = H(I2[t])
        r1[t] = r1[t] < 0 ? 0 : r1[t]
        r2[t] = r2[t] < 0 ? 0 : r2[t]

        # Update synaptic gating variables using Euler method with gamma_NMDA
        S1[t+1] = S1[t] + (- S1[t] / τ_NMDA + (1 - S1[t]) * γ * r1[t] / 1000) * dt
        S2[t+1] = S2[t] + (- S2[t] / τ_NMDA + (1 - S2[t]) * γ * r2[t] / 1000) * dt
    end

    # Calculate mean rates and gating variables with sliding window
    time_window = Int(50 / dt)                          # time window 50 ms
    sliding_step = Int(5 / dt)                          # sliding window step 5 ms
    r1_smooth = [mean(r1[1:time_window])]
    r2_smooth = [mean(r2[1:time_window])]
    S1_smooth = [mean(S1[1:time_window])]
    S2_smooth = [mean(S2[1:time_window])]
    for t in 1:Int(floor((n_time_steps - time_window) / sliding_step))
        start_idx = sliding_step * (t-1) + 1
        end_idx = sliding_step * (t-1) + time_window
        push!(r1_smooth, mean(r1[start_idx:end_idx]))
        push!(r2_smooth, mean(r2[start_idx:end_idx]))
        push!(S1_smooth, mean(S1[start_idx:end_idx]))
        push!(S2_smooth, mean(S2[start_idx:end_idx]))
    end

    return r1_smooth, r2_smooth, S1_smooth, S2_smooth
end

# Run simulations for different coherence values
results = Dict()
for coherence in coherence_values
    results[coherence] = run_simulation(coherence)
end

# Visualization using PlotlyJS

# Plot firing rates over time
traces_r1 = []
traces_r2 = []
colors = ["blue", "green", "red"]
for (i, coherence) in enumerate(coherence_values)
    r1_smooth, r2_smooth, _, _ = results[coherence]
    push!(traces_r1, scatter(
        x = dt*time_window:dt*sliding_step:dt*(n_time_steps - time_window),
        y = r1_smooth,
        mode = "lines",
        name = "Firing Rate Population 1 (coherence = $coherence)",
        line = attr(color = colors[i], width = 2)
    ))
    push!(traces_r2, scatter(
        x = dt*time_window:dt*sliding_step:dt*(n_time_steps - time_window),
        y = r2_smooth,
        mode = "lines",
        name = "Firing Rate Population 2 (coherence = $coherence)",
        line = attr(color = colors[i], width = 2)
    ))
end

# Create shape for stimulus period
stim_shape = [
    attr(
        type = "rect",
        xref = "x",
        yref = "paper",
        x0 = stim_start,
        y0 = 0,
        x1 = stim_end,
        y1 = 1,
        fillcolor = "rgba(0, 255, 0, 0.1)",
        line = attr(width = 0),
        layer = "below"
    )
]

# Create layout for time series plot
layout_time_series = Layout(
    title = "Wong and Wang Model Dynamics Over Time",
    xaxis = attr(title = "Time (ms)"),
    yaxis = attr(title = "Firing Rate (Hz)"),
    legend = attr(orientation = "v", x = 0.8, y = 0.9),
    shapes = stim_shape,
    height = 400
)

# Create figure for time series and display
fig_time_series = Plot(vcat(traces_r1..., traces_r2...), layout_time_series)

display(fig_time_series)

# Phase-plane Plot: Plot S1 vs S2
trace_phase_plane = scatter(
    x = S1_smooth,
    y = S2_smooth,
    mode = "lines+markers",
    marker = attr(
        color = Array(dt*time_window:dt*sliding_step:dt*(n_time_steps - time_window)),  # Color by time
        colorscale = "Viridis",
        size = 5,
        showscale = true,
        colorbar = attr(title = "Time (ms)")
    ),
    line = attr(color = "gray"),
    name = "Trajectory"
)

# Create layout for phase-plane plot
layout_phase_plane = Layout(
    xaxis_range=[0.0, 1.0], yaxis_range=[0.0, 1.0],
    title = "Phase-Plane Plot of Network Dynamics (S1 vs S2)",
    xaxis = attr(title = "Synaptic Gating Variable S1"),
    yaxis = attr(title = "Synaptic Gating Variable S2"),
    legend = attr(orientation = "h", x = 0.3, y = 1.1),
    width = 600,
    height = 500
)

# Create figure for phase-plane plot and display
fig_phase_plane = Plot([trace_phase_plane], layout_phase_plane)

display(fig_phase_plane)

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, legend, margin, shapes, template, title, xaxis, and yaxis"

data: [
  "scatter with fields line, marker, mode, name, type, x, and y"
]

layout: "layout with fields height, legend, margin, template, title, width, xaxis, and yaxis"

In [16]:
# Import necessary packages
using Random
using Statistics
using PlotlyJS

# Set random seed for reproducibility
Random.seed!(1234)

# Simulation Parameters
dt = 0.5                        # Time step (ms)
t_max = 3000.0                  # Total simulation time (ms)
time = 0.0:dt:t_max             # Time vector
n_time_steps = length(time)

# Model Parameters
# Time Constants
τ_NMDA = 100.0                  # NMDA synaptic time constant (ms)
τ_AMPA = 2.0                    # AMPA synaptic time constant (ms)

# Synaptic Efficacies
J_N = 0.2609                    # Recurrent NMDA excitation strength
J_N_cross = 0.0497              # Cross NMDA excitation

# NMDA Gain Factor
γ = 0.641                       # NMDA receptor gain factor

# External Inputs
I_0 = 0.3255                    # Baseline external input (nA)
μ0 = 30                         # Hz, mean input
J_Input = 5.2e-4                # Input excitation
coherence_levels = 0.0:0.01:0.1 # Coherence levels (0 to 1)
stim_start = 500.0              # Stimulus onset time (ms)
stim_end = 2000.0               # Stimulus offset time (ms)

# Noise Parameters
σ_I = 0.02                      # Reduced standard deviation of input noise (nA)

# Define transfer function H(I)
# F-I Curve Parameters (Excitatory Neurons)
a = 270.0                       # (Hz/nA)
b = 108.0                       # (Hz)
d = 0.154                       # (s)
function H(I)
    I_eff = a * I - b
    return I_eff / (1.0 - exp(-d * I_eff))
end

# Function to run a single simulation
function run_simulation(coherence)
    # Initialize variables
    S1 = zeros(n_time_steps)        # Synaptic gating variable for Population 1
    S2 = zeros(n_time_steps)        # Synaptic gating variable for Population 2
    r1 = zeros(n_time_steps)        # Firing rate of Population 1
    r2 = zeros(n_time_steps)        # Firing rate of Population 2
    I1 = zeros(n_time_steps)        # Total input current to Population 1
    I2 = zeros(n_time_steps)        # Total input current to Population 2

    # Stimulus input to Population 1&2
    I_stim_1 = J_Input * μ0 * (1 + coherence)
    I_stim_2 = J_Input * μ0 * (1 - coherence)

    # Precompute external inputs
    I_ext1 = fill(I_0, n_time_steps)
    I_ext2 = fill(I_0, n_time_steps)
    stim_indices = findall(t -> t >= stim_start && t <= stim_end, time)
    I_ext1[stim_indices] .+= I_stim_1
    I_ext2[stim_indices] .+= I_stim_2

    # Simulate the network dynamics
    I_noise1 = σ_I * randn()
    I_noise2 = σ_I * randn()
    S1[1] = S2[1] = 0.1
    decision_made = false
    decision_time = 0.0
    choice = 0

    for t in 1:n_time_steps-1
        # Compute noise terms
        I_noise1 = I_noise1 + dt / τ_AMPA * (-I_noise1) + sqrt(dt/τ_AMPA) * σ_I * randn()
        I_noise2 = I_noise2 + dt / τ_AMPA * (-I_noise2) + sqrt(dt/τ_AMPA) * σ_I * randn()

        # Compute total input currents
        I1[t] = J_N * S1[t] - J_N_cross * S2[t] + I_ext1[t] + I_noise1
        I2[t] = J_N * S2[t] - J_N_cross * S1[t] + I_ext2[t] + I_noise2

        # Compute firing rates
        r1[t] = H(I1[t])
        r2[t] = H(I2[t])
        r1[t] = r1[t] < 0 ? 0 : r1[t]
        r2[t] = r2[t] < 0 ? 0 : r2[t]

        # Update synaptic gating variables using Euler method with gamma_NMDA
        S1[t+1] = S1[t] + (- S1[t] / τ_NMDA + (1 - S1[t]) * γ * r1[t] / 1000) * dt
        S2[t+1] = S2[t] + (- S2[t] / τ_NMDA + (1 - S2[t]) * γ * r2[t] / 1000) * dt

        # Check for decision
        if !decision_made && (r1[t] > 15 || r2[t] > 15)
            decision_made = true
            decision_time = time[t]
            choice = r1[t] > r2[t] ? 1 : 2
        end
    end

    return decision_time, choice
end

# Run multiple simulations for different coherence levels
n_simulations = 100
decision_times = Dict{Float64, Vector{Float64}}()
choices = Dict{Float64, Vector{Int}}()
for coherence in coherence_levels
    decision_times[coherence] = Float64[]
    choices[coherence] = Int[]
    for _ in 1:n_simulations
        decision_time, choice = run_simulation(coherence)
        push!(decision_times[coherence], decision_time)
        push!(choices[coherence], choice)
    end
end

# Calculate psychometric and chronometric curves
psychometric_curve = [mean(choices[coherence] .== 1) for coherence in coherence_levels]
chronometric_curve = [mean(decision_times[coherence]) for coherence in coherence_levels]

# Plot psychometric curve
trace_psychometric = scatter(
    x = coherence_levels,
    y = psychometric_curve,
    mode = "lines+markers",
    name = "Psychometric Curve",
    line = attr(color = "blue", width = 2)
)

layout_psychometric = Layout(
    title = "Psychometric Curve",
    xaxis = attr(title = "Coherence"),
    yaxis = attr(title = "Probability of Choosing Population 1"),
    height = 400
)

fig_psychometric = Plot([trace_psychometric], layout_psychometric)
display(fig_psychometric)

# Plot chronometric curve
trace_chronometric = scatter(
    x = coherence_levels,
    y = chronometric_curve,
    mode = "lines+markers",
    name = "Chronometric Curve",
    line = attr(color = "red", width = 2)
)

layout_chronometric = Layout(
    title = "Chronometric Curve",
    xaxis = attr(title = "Coherence"),
    yaxis = attr(title = "Decision Time (ms)"),
    height = 400
)

fig_chronometric = Plot([trace_chronometric], layout_chronometric)
display(fig_chronometric)

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, margin, template, title, xaxis, and yaxis"

data: [
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, margin, template, title, xaxis, and yaxis"